[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Methods


## What you will be able to do

Write methods that belong to an object: ones that compute and return a value, ones that change the
object, and ones that call each other. And read `north.average()` as the call it actually is.


## The idea

### The problem

The **Your First Class** notebook defined one method, `average`, and called it. Two things about
that call went unexplained.

`average` was written as `def average(self)`, taking one parameter, and called as
`north.average()`, passing none. The **Your First Class** notebook established that Python supplies
the object as the first argument, so that much follows. What it did not say is what
`north.average` is when you stop before the parentheses. It is a value, it can be printed and
stored in a variable, and knowing what it is explains a whole family of errors.

The second thing is a design question, and it causes more trouble than the syntax ever will.

You have a station and a new reading to add to it. Two reasonable methods are available.
`station.add_reading(-5.2)` changes the station and hands nothing back, while
`station.with_reading(-5.2)` changes nothing and hands back a second station. Both are used in real
code, and the **Lists** notebook showed the same split in the standard library: `readings.sort()`
reorders the list and returns `None`, and `sorted(readings)` returns a new list.

Confusing the two produces a specific bug. Write `readings = readings.sort()` and `readings` is now
`None`, and the error arrives on some later line that looks innocent. The same mistake is available
on every method you write.

### What a method is

> A **method** is a function defined inside a class. Reaching it through an object,
> `north.average`, produces a **bound method**: the same function, with that object already
> attached as the first argument. Calling it, `north.average()`, runs the function with `self` set
> to `north`.

### Why it works that way

The word to hold on to is *attached*. `Station.average` is a plain function that needs a station
passed to it. `north.average` is that same function with `north` already supplied. So
`north.average()` and `Station.average(north)` are two spellings of one call, and the notebook
shows them side by side rather than asking you to believe it.

Because the attaching happens when you reach for the attribute and not when you call it,
`north.average` on its own is a finished value. You can put it in a variable, put several in a
list, and call them later. That is also why leaving off the parentheses does not raise where you
made the mistake: you get the method itself, which is a valid object, and the `TypeError` comes
later from whatever tried to use it as a number.

For the design question, the useful rule is that a method should change the object or return a
result, and not both. A method that changes the object returns `None`, which is what a function
with no `return` gives you anyway. A method that computes returns the answer and leaves the object
exactly as it found it. When a method does both, every caller has to remember which half they
wanted.

One more thing follows from all of this. A method calling another method on the same object has to
go through `self`. Inside `report`, the name `average` on its own does not exist; `self.average()`
does. Methods are not in scope inside each other, and that catches everybody once.

### Where you will meet this

The standard library follows the rule closely enough to predict. `list.append` and `list.sort`
change the list and return `None`. `str.upper` and `str.strip` return new strings and change
nothing, because a string cannot be changed. `dict.get` returns a value; `dict.update` changes the
dictionary. When you meet an unfamiliar method, the question "does this hand something back, or
change the thing" usually has an answer you can guess.

### What this notebook covers

What `north.average` is before it is called, and the two spellings of one call. Methods that
return, methods that change, and why not both. Methods calling methods through `self`. A method
that returns a new object rather than changing this one.

### A first look

One method, called two ways. There is nothing to run yet: read it, and read the output underneath
it.

```python
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def coldest(self):
        return min(self.readings)


north = Station("Tromso", [-4.1, -2.6, -3.8])
print(north.coldest())
print(Station.coldest(north))
```

```
-4.1
-4.1
```

The second line is what the first line becomes. Everything else in this notebook follows from that.


## Setup

One import, and the class the whole notebook works with.

- `mean` averages a list of numbers, used by the `average` method

`Station` is defined once here with five methods, and the sections below take them one at a time.
Two of them, `add_reading` and `in_fahrenheit`, are the same job done in the two different styles
this notebook is about.

**Run this cell before the rest of the notebook.**


In [1]:
from statistics import mean


class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def average(self):
        return round(mean(self.readings), 1)

    def coldest(self):
        return min(self.readings)

    def add_reading(self, value):
        self.readings.append(value)

    def report(self):
        return (f"{self.name}: mean {self.average()} {self.unit}, "
                f"coldest {self.coldest()} {self.unit}")

    def in_fahrenheit(self):
        if self.unit == "F":
            return Station(self.name, list(self.readings), "F")
        return Station(self.name, [round(r * 9 / 5 + 32, 1) for r in self.readings], "F")


north = Station("Tromso", [-4.1, -2.6, -3.8])

print(north.report())


Tromso: mean -3.5 C, coldest -4.1 C


## Worked examples

### Reached on the class, and reached on the object

The same name gives you two different things depending on what you reach through.


In [2]:
print("Station.average is a", type(Station.average).__name__)
print("north.average is a  ", type(north.average).__name__)

print()
print("bound to north:      ", north.average.__self__ is north)
print("same function inside:", north.average.__func__ is Station.average)


Station.average is a function
north.average is a   method

bound to north:       True
same function inside: True


Reached on the class, `average` is a plain function that has not been given a station yet.

Reached on the object, it is a **bound method**: a small wrapper holding two things. `__self__` is
the object it was reached through, and `__func__` is the function from the class. They are not
copies; `__func__ is Station.average` is `True`, so there is one function and every station shares
it.

You will not often write `__self__` or `__func__`. They are here because they make the wrapper
visible, and once you have seen what a bound method is holding, the errors later in this notebook
stop being surprising.

### Two spellings of one call

Calling the bound method runs the function with `__self__` supplied as the first argument. Passing
the station by hand does the same thing.


In [3]:
print("north.average():       ", north.average())
print("Station.average(north):", Station.average(north))
print("the same:", north.average() == Station.average(north))


north.average():        -3.5
Station.average(north): -3.5
the same: True


`north.average()` is the form to write. `Station.average(north)` is what it means.

This is the explanation for an error the **Why Classes** notebook showed without unpacking:
`Station.report()` raises `TypeError` naming `self` as the missing argument, because reaching
through the class attaches nothing, so the first argument is simply absent.

### A bound method is a value

Because the object is attached at the moment you reach for the attribute, `north.average` is
finished before any call happens. It can be stored and passed around like any other value.


In [4]:
saved = north.average
print("stored, then called:", saved())

for method in [north.average, north.coldest]:
    print(f"  {method.__name__:<8} -> {method()}")


stored, then called: -3.5
  average  -> -3.5
  coldest  -> -4.1


`saved` still knows which station it belongs to, because the station is inside it.

This is worth knowing beyond the trivia. Passing `north.average` to something that will call it
later, such as a sort key or a callback, is common, and it works because the object travels with
the method.

### A method that changes the object

`add_reading` appends to the list on `self`. It has no `return`, so it hands back `None`.


In [5]:
print("readings before:     ", north.readings)

returned = north.add_reading(-5.2)

print("readings after:      ", north.readings)
print("add_reading returned:", returned)


readings before:      [-4.1, -2.6, -3.8]
readings after:       [-4.1, -2.6, -3.8, -5.2]
add_reading returned: None


The station changed and nothing came back. That is the correct shape for a method of this kind, and
it is what `list.append` does.

### A method that returns a value

`average` computes and returns. It reads `self.readings` and does not touch it.


In [6]:
print("average() returned:", north.average())
print("readings after:    ", north.readings)
print("called again:      ", north.average())


average() returned: -3.9
readings after:     [-4.1, -2.6, -3.8, -5.2]
called again:       -3.9


Calling it twice gives the same answer, because nothing about the station moved. A method you can
call repeatedly without consequence is much easier to reason about, and much easier to test.

### Why not both

A method that changes the object **and** returns something forces every caller to know which half
they are getting.

Consider a `trim` method that removes readings below a limit and also returns how many it removed.
The call `count = station.trim(-5)` reads as though it produced a number and nothing else, and a
reader has to know the method to know the station was modified. The call `station.trim(-5)` on its
own reads as though it changed something, and silently discards a number that may have mattered.

Neither reading is wrong, which is the problem. Splitting it into `station.trim(-5)`, which changes
and returns `None`, and `station.count_below(-5)`, which returns and changes nothing, means the
call site says which one happened.

The exception is small and well known: `dict.pop` and `list.pop` remove an item and return it,
because removing and getting are one operation there and everybody expects it. Follow the rule
until you have a reason that good.

### Methods calling methods

`report` calls two other methods on the same object. Each call goes through `self`.


In [7]:
print(north.report())


Tromso: mean -3.9 C, coldest -5.2 C


Inside `report`, the bare name `average` does not exist. Methods do not become local names inside
one another, so `self.average()` is the only way to reach it. This is one of the most common
mistakes when writing a class, and it is in the errors below.

`self.average()` also means `report` gets whatever `average` currently is. If `average` is changed,
`report` follows without being edited.

### A method that returns a new object

`in_fahrenheit` is the other style: it changes nothing and returns a second station.


In [8]:
warm = north.in_fahrenheit()

print("north:", north.readings, north.unit)
print("warm: ", warm.readings, warm.unit)
print("same object:", warm is north)


north: [-4.1, -2.6, -3.8, -5.2] C
warm:  [24.6, 27.3, 25.2, 22.6] F
same object: False


Two stations now exist, and the original is untouched. `list(self.readings)` in the `unit == "F"`
branch is deliberate: without it the new station would share the original's list, and appending to
one would show up in the other, which is the aliasing trap from **Lists** and from **Why Classes**.

This style costs memory and buys safety. A method that returns a new object can be called anywhere
without checking what else holds a reference to the original, which is why `str` methods all work
this way.

Choose per method, and say which you chose in the name. `add_reading` sounds like it changes
something. `in_fahrenheit` sounds like it produces something. Names carrying that information are
worth more than a comment.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/03-methods-solutions.ipynb).

**1.** Add a `warmest()` method to `Station`, returning the highest reading. Call it on a station
and print the result.


In [9]:
# your code here


**2.** Add a `drop_below(limit)` method that removes every reading below `limit` from the station
and returns `None`. Print the readings before and after, and print what the call returned.


In [10]:
# your code here


**3.** Call `coldest` on a station both ways, through the object and through the class, and print
whether the two results are equal.


In [11]:
# your code here


**4.** This `report` raises. Say in a comment what the error will be, then fix it and run it.

```python
class Broken:
    def __init__(self, readings):
        self.readings = readings

    def average(self):
        return round(mean(self.readings), 1)

    def report(self):
        return f"mean {average()}"
```


In [12]:
# your code here


**5.** Add a `rounded()` method returning a **new** `Station` whose readings are rounded to whole
numbers, leaving the original alone. Print both stations' readings and whether they are the same
object.


In [13]:
# your code here


**6.** No code for this one. For each method below, write a comment saying whether it should change
the object and return `None`, or return a value and change nothing, and one sentence of why.

- `station.rename(new_name)`
- `station.days_below(0)`
- `station.merge(other_station)`


In [14]:
# your code here


## Common errors

### TypeError: a method defined without `self`

Every method reached through an object is handed that object. A method whose `def` has no
parameters has nowhere to put it.


In [15]:
class NoSelf:
    def __init__(self, name):
        self.name = name

    def shout():
        return "hello"


NoSelf("Tromso").shout()


TypeError: NoSelf.shout() takes 0 positional arguments but 1 was given

`takes 0 positional arguments but 1 was given` describes exactly what happened: you passed none,
Python added the object, and the count went to one.

This is the same arithmetic as the `__init__` error in **Your First Class**, and it has the same
fix. Every method that will be called on an object takes `self` first.

### NameError: a sibling method called without `self.`

Methods are attributes of the class, not names in the surrounding code. Inside one method, another
method's bare name is not defined.


In [16]:
class Bare:
    def __init__(self, readings):
        self.readings = readings

    def average(self):
        return round(mean(self.readings), 1)

    def report(self):
        return f"mean {average()}"


Bare([-4.1, -2.6]).report()


NameError: name 'average' is not defined

`name 'average' is not defined`, in a class that plainly defines `average`.

The class body is not a scope that methods can see into, which is why `self.average()` is required
rather than merely preferred. It reads as a rule to memorize and it is really the **Scope**
notebook's rule unchanged: `average` is not a local, not an enclosing name, and not a global, so
Python has nowhere left to look.


In [17]:
class Fixed:
    def __init__(self, readings):
        self.readings = readings

    def average(self):
        return round(mean(self.readings), 1)

    def report(self):
        return f"mean {self.average()}"


print(Fixed([-4.1, -2.6]).report())


mean -3.3


### AttributeError: a method name that does not exist

A misspelled method is not caught until the line runs, because the lookup happens at that moment.


In [18]:
north.avarage()


AttributeError: 'Station' object has no attribute 'avarage'

`dir(object)` lists everything reachable on it. Filtering out the names that begin with an
underscore leaves the attributes and methods you defined, which is usually enough to spot the
typo.


In [19]:
print([name for name in dir(north) if not name.startswith("_")])


['add_reading', 'average', 'coldest', 'in_fahrenheit', 'name', 'readings', 'report', 'unit']


### TypeError: the parentheses were left off

`north.average` is the method. `north.average()` is what it returns. Using the first where you
meant the second does not raise on that line, because a bound method is a valid value. The error
comes from whatever tries to use it.


In [20]:
if north.average > 0:
    print("warm")


TypeError: '>' not supported between instances of 'method' and 'int'

`'>' not supported between instances of 'method' and 'int'` names the type that arrived. Whenever
`method` appears where you expected a number or a string, this is the reason.

### The quiet one: assigning the result of a method that changes the object

`add_reading` returns `None`. Assigning that result over the station replaces it with `None`, and
nothing raises until something later tries to use it.


In [21]:
station = Station("Bodo", [-2.6, -1.9])
station = station.add_reading(-3.3)

print("station is now:", station)
print("type:          ", type(station).__name__)


station is now: None
type:           NoneType


The reading was added. Then the station was thrown away and replaced with what the method returned,
which was nothing.

The failure surfaces on whichever line next uses `station`, which may be far away and will name
`NoneType` rather than anything you recognize.


In [22]:
station.average()


AttributeError: 'NoneType' object has no attribute 'average'

`'NoneType' object has no attribute 'average'` almost always means this: a method that changes
something was assigned as though it returned something.

This is the same trap as `readings = readings.sort()` from **Lists**, and the same rule avoids it.
A method that changes the object is called on its own line, and its result is not used.


## Recap

- A method is a function in a class. Reaching it through an object gives a bound method, with the
  object already attached.
- `north.average()` and `Station.average(north)` are the same call.
- A bound method is a value, so it can be stored, listed and passed to something that calls it
  later.
- A method that changes the object returns `None`. A method that computes returns the answer and
  changes nothing.
- Doing both makes the call site ambiguous, so split it into two methods.
- One method calls another through `self`. A bare method name inside a class is not defined.
- A method returning a new object leaves the original safe, and must copy any list it carries over.
- Name the method after which kind it is: `add_reading` changes, `in_fahrenheit` produces.
- A method defined without `self` fails with an argument count one higher than you passed.
- `dir(object)` lists what is reachable, which finds a misspelled method name.
- Assigning the result of a method that returns `None` replaces your object with `None`, and the
  error arrives somewhere else.


## What is next

The **Dunder Methods** notebook. Printing a station currently shows its class and a memory address,
which is no use to anybody, and comparing two stations with `==` says they differ even when every
attribute matches. Both are fixed by methods with double underscores in their names, which Python
calls for you in situations where your object is being printed, compared or measured.


---

&#8592; **Previous:** [Your First Class](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/02-your-first-class.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
